# A Quick Introduction to Optuna

This Jupyter notebook goes through the basic usage of Optuna.

- Install Optuna
- Write a training algorithm that involves hyperparameters
  - Read train/valid data
  - Define and train model
  - Evaluate model
- Use Optuna to tune the hyperparameters (hyperparameter optimization, HPO)
- Visualize HPO

## Install `optuna`

Optuna can be installed via `pip` or `conda`.

In [1]:
!pip install --quiet optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 15.7 MB/s eta 0:00:00


In [2]:
import optuna

optuna.__version__

'4.4.0'

## Optimize Hyperparameters

### Define a simple scikit-learn model

We start with a simple random forest model to classify flowers in the Iris dataset. We define a function called `objective` that encapsulates the whole training process and outputs the accuracy of the model.

In [3]:
import sklearn.datasets
import sklearn.ensemble
import sklearn.model_selection


def objective():
    iris = sklearn.datasets.load_iris()  # Prepare the data.

    clf = sklearn.ensemble.RandomForestClassifier(n_estimators=5, max_depth=3)  # Define the model.

    return sklearn.model_selection.cross_val_score(
        clf, iris.data, iris.target, n_jobs=-1, cv=3
    ).mean()  # Train and evaluate the model.


print("Accuracy: {}".format(objective()))

Accuracy: 0.96


### Optimize hyperparameters of the model

The hyperparameters of the above algorithm are `n_estimators` and `max_depth` for which we can try different values to see if the model accuracy can be improved. The `objective` function is modified to accept a trial object. This trial has several methods for sampling hyperparameters. We create a study to run the hyperparameter optimization and finally read the best hyperparameters.

In [4]:
import optuna


def objective(trial):
    iris = sklearn.datasets.load_iris()

    n_estimators = trial.suggest_int("n_estimators", 2, 20)
    max_depth = int(trial.suggest_float("max_depth", 1, 32, log=True))

    clf = sklearn.ensemble.RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth)

    return sklearn.model_selection.cross_val_score(
        clf, iris.data, iris.target, n_jobs=-1, cv=3
    ).mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

trial = study.best_trial

print("Accuracy: {}".format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

[I 2025-07-07 08:17:14,505] A new study created in memory with name: no-name-5af57bc6-e975-4b8e-a443-2de08e9c59fc
[I 2025-07-07 08:17:14,581] Trial 0 finished with value: 0.96 and parameters: {'n_estimators': 10, 'max_depth': 20.02387779161834}. Best is trial 0 with value: 0.96.
[I 2025-07-07 08:17:14,622] Trial 1 finished with value: 0.9466666666666667 and parameters: {'n_estimators': 4, 'max_depth': 4.9786363465636425}. Best is trial 0 with value: 0.96.
[I 2025-07-07 08:17:14,673] Trial 2 finished with value: 0.9466666666666667 and parameters: {'n_estimators': 6, 'max_depth': 17.12341931945286}. Best is trial 0 with value: 0.96.
[I 2025-07-07 08:17:14,723] Trial 3 finished with value: 0.96 and parameters: {'n_estimators': 7, 'max_depth': 5.8668403863512975}. Best is trial 0 with value: 0.96.
[I 2025-07-07 08:17:14,755] Trial 4 finished with value: 0.7600000000000001 and parameters: {'n_estimators': 2, 'max_depth': 1.2754871868550757}. Best is trial 0 with value: 0.96.
[I 2025-07-07 0

Accuracy: 0.9666666666666667
Best hyperparameters: {'n_estimators': 20, 'max_depth': 7.829543189605181}


It is possible to condition hyperparameters using Python `if` statements. We can for instance include another classifier, a support vector machine, in our HPO and define hyperparameters specific to the random forest model and the support vector machine.

In [5]:
import sklearn.svm


def objective(trial):
    iris = sklearn.datasets.load_iris()

    classifier = trial.suggest_categorical("classifier", ["RandomForest", "SVC"])

    if classifier == "RandomForest":
        n_estimators = trial.suggest_int("n_estimators", 2, 20)
        max_depth = int(trial.suggest_float("max_depth", 1, 32, log=True))

        clf = sklearn.ensemble.RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth
        )
    else:
        c = trial.suggest_float("svc_c", 1e-10, 1e10, log=True)

        clf = sklearn.svm.SVC(C=c, gamma="auto")

    return sklearn.model_selection.cross_val_score(
        clf, iris.data, iris.target, n_jobs=-1, cv=3
    ).mean()


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

trial = study.best_trial

print("Accuracy: {}".format(trial.value))
print("Best hyperparameters: {}".format(trial.params))

[I 2025-07-07 08:18:01,753] A new study created in memory with name: no-name-d2735901-a301-47e1-aa5a-eb2e50e6423f
[I 2025-07-07 08:18:01,907] Trial 0 finished with value: 0.7733333333333333 and parameters: {'classifier': 'RandomForest', 'n_estimators': 20, 'max_depth': 1.0136266397882954}. Best is trial 0 with value: 0.7733333333333333.
[I 2025-07-07 08:18:01,938] Trial 1 finished with value: 0.9666666666666667 and parameters: {'classifier': 'SVC', 'svc_c': 0.7322275412121207}. Best is trial 1 with value: 0.9666666666666667.
[I 2025-07-07 08:18:02,050] Trial 2 finished with value: 0.86 and parameters: {'classifier': 'RandomForest', 'n_estimators': 17, 'max_depth': 1.2812683856977445}. Best is trial 1 with value: 0.9666666666666667.
[I 2025-07-07 08:18:02,158] Trial 3 finished with value: 0.9466666666666667 and parameters: {'classifier': 'RandomForest', 'n_estimators': 16, 'max_depth': 16.72960590245643}. Best is trial 1 with value: 0.9666666666666667.
[I 2025-07-07 08:18:02,250] Trial 

Accuracy: 0.9866666666666667
Best hyperparameters: {'classifier': 'SVC', 'svc_c': 4.243072035314304}


### Plotting the study

Plotting the optimization history of the study.

In [6]:
optuna.visualization.plot_optimization_history(study)

Plotting the accuracies for each hyperparameter for each trial.

In [7]:
optuna.visualization.plot_slice(study)

Plotting the accuracy surface for the hyperparameters involved in the random forest model.

In [8]:
optuna.visualization.plot_contour(study, params=["n_estimators", "max_depth"])